# Session 9: LLM-Supported Narrative Explanation (RQ4 / Format F4)

This notebook implements **F4**, the LLM-supported narrative explanation format from the thesis proposal (RQ4). For each of the 7 models, it takes the already-generated, rule-based facts (the four fuzzy labels and the narrative paragraph from notebook 06) and asks a small, locally-run language model to **rephrase** them into more natural prose — without introducing any new facts, numbers, comparisons, or judgments.

**Why a local model instead of a cloud API:** this avoids any API key or billing dependency for a one-off thesis artifact, and keeps the notebook fully reproducible by anyone with [Ollama](https://ollama.com) installed, with no account needed. The tradeoff is lower fluency/completeness than a frontier model — discussed as a limitation below.

**Prerequisites to run this notebook:**
1. Install Ollama: https://ollama.com
2. Pull the model: `ollama pull llama3`
3. Start the server: `ollama serve` (it may already be running as a background service)

**Known limitations (both observed empirically during actual runs of this notebook, not just theoretical):**
1. The model sometimes *omits* a fact (e.g. drops the directional-accuracy sentence) rather than fabricating one. The faithfulness check below only screens for fabricated numbers, not omissions.
2. The model can quietly *amplify the strength* of a claim without using any number, which the numeric check cannot catch. Observed example from an actual run: for Naive, the template says "a slight tendency to overpredict demand," and the LLM rewrote this as "consistently overpredicts demand by a significant margin" — a clear violation of rule 3 (don't change the strength of a claim) that produced no flagged number.

Both points mean the side-by-side comparison printed near the end of this notebook must be read manually before any of these narratives are used in the user study — the automated check is a screening aid for fabricated numbers only, not a guarantee of faithfulness.

In [1]:
import json
import re
import time

import pandas as pd
import requests

In [2]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "llama3"
TEMPERATURE = 0.2

## Prompt template

Kept here as plain variables, not buried inside a function, so they can be quoted directly in the methodology chapter.

In [3]:
SYSTEM_PROMPT = """You are rewriting a short, already-correct forecast explanation into \
more natural, fluent prose for a non-technical reader.

Strict rules:
1. Use ONLY the facts given to you below. Do not add any new facts, numbers, \
comparisons, judgments, or qualifiers that are not explicitly present in the input.
2. Do not invent or estimate any numeric value, percentage, or count, even if it \
seems reasonable to infer one.
3. Do not change the meaning, direction, or strength of any claim (for example, do \
not turn "slight overprediction" into "significant overprediction").
4. Do not compare this model to any other model.
5. Output exactly one paragraph, 2-4 sentences, in plain English. No headers, no \
bullet points, no preamble such as "Here is the rewritten text" — output ONLY the \
paragraph itself."""

USER_PROMPT_TEMPLATE = """Model name: {model}

Established facts (already verified — do not alter their meaning):
- Error magnitude category: {mae_label}
- Bias direction category: {mpe_label}
- Percentage-error category: {mape_label}
- Directional-accuracy category: {da_label}
- Existing explanation: "{narrative}"

Rewrite the existing explanation above into more natural, fluent prose using only \
the facts listed. Do not add anything new."""

## Load notebook 06's rule-based narratives

In [4]:
narratives = pd.read_csv("../src/data/session06_narratives.csv")
narratives[["Model", "MAE_Label", "MPE_Label", "MAPE_Label", "DA_Label", "Narrative"]]

,Model,MAE_Label,MPE_Label,MAPE_Label,DA_Label,Narrative
0,Naive,high error,slight overprediction,high mape,low DA,Naive is among the weaker models in this evalu...
1,Seasonal Naive,medium error,neutral,medium mape,medium DA,Seasonal Naive shows moderate forecasting accu...
2,Linear Regression,low error,slight overprediction,medium mape,low DA,Linear Regression performs well in terms of ab...
3,ETS,medium error,slight overprediction,medium mape,medium DA,ETS shows moderate forecasting accuracy. The m...
4,HWES (damped),medium error,slight overprediction,medium mape,medium DA,HWES (damped) shows moderate forecasting accur...
5,SARIMA,medium error,slight overprediction,medium mape,medium DA,SARIMA shows moderate forecasting accuracy. Th...
6,Prophet,low error,slight overprediction,low mape,medium DA,Prophet is one of the stronger models in this ...


## Call the local LLM — one constrained rewrite per model

In [5]:
def call_ollama(system_prompt, user_prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "stream": False,
            "options": {"temperature": TEMPERATURE},
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["message"]["content"].strip()

## Faithfulness check

Lightweight screening only: flags any number in the LLM output that wasn't present in the source narrative it was based on. Since the rule-based narratives contain no numeric digits at all, any number appearing in the LLM output is automatically suspect.

In [6]:
NUMBER_RE = re.compile(r"\d+\.?\d*")

def extract_numbers(text):
    return set(NUMBER_RE.findall(text))

def check_faithfulness(llm_text, source_text):
    output_numbers = extract_numbers(llm_text)
    input_numbers = extract_numbers(source_text)
    return sorted(output_numbers - input_numbers)

In [7]:
results = []
for _, row in narratives.iterrows():
    user_prompt = USER_PROMPT_TEMPLATE.format(
        model=row["Model"],
        mae_label=row["MAE_Label"],
        mpe_label=row["MPE_Label"],
        mape_label=row["MAPE_Label"],
        da_label=row["DA_Label"],
        narrative=row["Narrative"],
    )
    try:
        llm_narrative = call_ollama(SYSTEM_PROMPT, user_prompt)
    except requests.exceptions.ConnectionError:
        raise RuntimeError(
            f"Could not reach Ollama at {OLLAMA_URL}. Make sure Ollama is installed and "
            "running (`ollama serve`) and that the model is pulled (`ollama pull llama3`)."
        )

    flagged_numbers = check_faithfulness(llm_narrative, row["Narrative"])

    results.append({
        "model": row["Model"],
        "mae_label": row["MAE_Label"],
        "mpe_label": row["MPE_Label"],
        "mape_label": row["MAPE_Label"],
        "da_label": row["DA_Label"],
        "template_narrative": row["Narrative"],
        "llm_narrative": llm_narrative,
        "faithfulness_flag": len(flagged_numbers) > 0,
        "flagged_numbers": flagged_numbers,
    })
    print(f"\u2713 {row['Model']}")
    time.sleep(0.2)

results_df = pd.DataFrame(results)
results_df[["model", "faithfulness_flag", "flagged_numbers"]]

✓ Naive


✓ Seasonal Naive


✓ Linear Regression


✓ ETS


✓ HWES (damped)


✓ SARIMA


✓ Prophet


,model,faithfulness_flag,flagged_numbers
0,Naive,False,[]
1,Seasonal Naive,False,[]
2,Linear Regression,False,[]
3,ETS,False,[]
4,HWES (damped),False,[]
5,SARIMA,False,[]
6,Prophet,False,[]


## Side-by-side comparison (template vs. LLM)

Manually skim this for omissions, not just the automated faithfulness flags above.

In [8]:
for r in results:
    print(f"=== {r['model']} ===")
    print("Template:", r["template_narrative"])
    print("LLM:     ", r["llm_narrative"])
    if r["faithfulness_flag"]:
        print(f"\u26a0 flagged numbers not present in source: {r['flagged_numbers']}")
    print()

=== Naive ===
Template: Naive is among the weaker models in this evaluation. The model has high absolute error with a slight tendency to overpredict demand. Its percentage error is high, limiting usefulness for demand-sensitive decisions. It struggles to correctly identify whether demand will rise or fall. This makes it less suited for scheduling decisions that depend on knowing whether demand will rise or fall.
LLM:      Naive is one of the weaker models in this evaluation. It consistently overpredicts demand by a small margin, resulting in high absolute error. This tendency to overestimate demand also leads to high percentage errors, making it less reliable for decisions that rely on accurate demand forecasts. Furthermore, Naive struggles to accurately predict whether demand will increase or decrease, which can be problematic when scheduling decisions depend on knowing the direction of demand changes.

=== Seasonal Naive ===
Template: Seasonal Naive shows moderate forecasting accurac

## Save output

In [9]:
with open("../src/data/session09_llm_narratives.json", "w") as f:
    json.dump(results, f, indent=2)

n_flagged = sum(r["faithfulness_flag"] for r in results)
print(f"Saved session09_llm_narratives.json \u2014 {n_flagged} of {len(results)} models flagged for manual review.")

Saved session09_llm_narratives.json — 0 of 7 models flagged for manual review.
